In [15]:
import pandas as pd
import numpy as np
import pycountry_convert as pc

from bokeh.plotting import output_notebook
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.transform import factor_cmap
from bokeh.palettes import Set2

output_notebook()

Loading BokehJS ...

In [23]:
def country_to_continent(country_name):
    try:
        country_code = pc.country_name_to_country_alpha2(country_name)
        continent_code = pc.country_alpha2_to_continent_code(country_code)
        continent_name = pc.convert_continent_code_to_continent_name(continent_code)
        return continent_name
    except KeyError:
        pass


def merge_datasets(df: pd.DataFrame) -> pd.DataFrame:
    df["f1"]  = 2 * df.TP / (2 * df.TP + df.FP + df.FN)
    df["continent"] = df["country"].apply(country_to_continent)
    df = df[~df.continent.isna()]

    hdi_df = pd.read_csv("../../data/human_development_index.csv")[["iso3", "hdi_2020"]]
    df = pd.merge(left=df, right=hdi_df, how="left", left_on="gid", right_on="iso3")
    df = df[~df.hdi_2020.isna()].rename(columns={"hdi_2020": "hdi"}).drop(columns="iso3")

    df['size'] = np.log1p(df['pixel_count'])

    return df


def plot_scatter(df: pd.DataFrame):
    continents = df['continent'].unique().tolist()
    palette = Set2[max(3, len(continents))]

    source = ColumnDataSource.from_df(df)


    fig = figure(
        x_axis_label='Human Development Index', 
        y_axis_label='F1',
        width=980
    )
    fig.scatter(
        x="hdi",
        y="f1", 
        source=source,
        alpha=0.8,
        color=factor_cmap('continent', palette=palette, factors=continents),
        legend_field='continent',
        size="size"
    )

    hover = HoverTool(tooltips=[
        ("Country", "@country"),
        ("F1", "@f1"),
        ("Pixel Count", "@pixel_count"),
    ])

    fig.legend.location = "bottom_right"

    fig.add_tools(hover)
    show(fig)

## 1% Threshold

In [24]:
df = pd.read_parquet("../../results/global_1_percent_thresh")
df = merge_datasets(df)
plot_scatter(df)

*size of the circles is `log(total pixel count)`

## 20% Threshold

In [25]:
df = pd.read_parquet("../../results/global_20_percent_thresh")
df = merge_datasets(df)
plot_scatter(df)